# Module 02 — Lesson 04
# Functions

---

> Every time you copy-paste code, you create a bug in two places instead of one.

Look back at the last three lessons. How many times did you write grade classification logic? How many times did you compute a mean? How many times did you handle `None` values? If you worked through the exercises, probably four or five times each.

**Functions** solve this. Write the logic once, give it a name, and call it from anywhere. Fix a bug in the function and it's fixed everywhere. Change behaviour in one place and it updates everywhere.

This is the **DRY principle** — Don't Repeat Yourself. It's not just about saving keystrokes. Repeated code is harder to read, harder to test, and dramatically harder to maintain. In data science, where you're constantly applying the same transformations to different columns and datasets, functions are the difference between clean pipelines and spaghetti notebooks.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Define and call functions using `def` and `return`
- Use positional, keyword, and default-value parameters
- Return multiple values from a single function
- Explain variable scope and why isolating logic in functions matters
- Write clear docstrings so your functions are self-documenting
- Use `lambda` for simple one-line functions in sorting and filtering
- Spot and fix the mutable default argument bug
- Build a reusable data science toolkit from scratch

---
## 1. Defining and Calling a Function

```python
def function_name(parameter1, parameter2):
    """One-line docstring: what this function does."""
    # function body — indented
    result = do_something(parameter1, parameter2)
    return result
```

- `def` signals the start of a function definition
- **Parameters** are the variables listed in the parentheses — placeholders for inputs
- **Arguments** are the actual values passed when you call the function
- `return` sends a value back to the caller. Without it, the function returns `None`.

In [ ]:
# The problem without functions — repeated logic
score1 = 88
if score1 >= 80: grade1 = "A"
elif score1 >= 60: grade1 = "B"
elif score1 >= 40: grade1 = "C"
else: grade1 = "F"

score2 = 55
if score2 >= 80: grade2 = "A"   # exact same logic, copy-pasted
elif score2 >= 60: grade2 = "B"
elif score2 >= 40: grade2 = "C"
else: grade2 = "F"

print(f"Score {score1} → {grade1},  Score {score2} → {grade2}")

In [ ]:
# The solution: define the logic ONCE as a function
def get_grade(score):
    """Return letter grade for a numeric score (0–100)."""
    if score >= 80:   return "A"
    elif score >= 60: return "B"
    elif score >= 40: return "C"
    else:             return "F"

# Call it as many times as needed
print(get_grade(88))    # A
print(get_grade(55))    # C
print(get_grade(32))    # F

# Apply to a whole list — one line
scores = [88, 55, 72, 32, 91, 65]
grades = [get_grade(s) for s in scores]
print(list(zip(scores, grades)))

In [ ]:
# Functions make fixing bugs trivially easy
# Suppose the threshold for A changes from 80 to 85.
# Without functions: you'd hunt down every place you copy-pasted that logic.
# With functions: change ONE line.

def get_grade_updated(score):
    """Return letter grade — A threshold raised to 85."""
    if score >= 85:   return "A"   # ← only change needed
    elif score >= 60: return "B"
    elif score >= 40: return "C"
    else:             return "F"

for s in scores:
    old = get_grade(s)
    new = get_grade_updated(s)
    changed = " ← changed" if old != new else ""
    print(f"  score {s:>3}: {old} → {new}{changed}")

---
## 2. Parameters and Arguments

### 2.1 Positional Arguments

Arguments are matched to parameters by **position** — the first argument goes to the first parameter.

In [ ]:
def compute_bmi(weight_kg, height_m):
    """Return BMI = weight / height²."""
    return round(weight_kg / height_m ** 2, 1)

# Positional: weight goes to weight_kg, height goes to height_m
bmi = compute_bmi(70, 1.75)
print(f"BMI: {bmi}")

# ❌ Order matters — wrong order gives wrong (or nonsensical) result
wrong_bmi = compute_bmi(1.75, 70)   # height treated as weight!
print(f"Wrong BMI (args reversed): {wrong_bmi}")

### 2.2 Keyword Arguments

Pass arguments by **name** to make calls self-documenting and order-independent.

In [ ]:
# Keyword arguments — explicitly name what each value means
bmi = compute_bmi(weight_kg=70, height_m=1.75)
print(f"BMI (keyword args): {bmi}")

# Order doesn't matter when using keyword args
bmi = compute_bmi(height_m=1.75, weight_kg=70)   # same result
print(f"BMI (reversed keywords): {bmi}")

# Mix positional and keyword — positional first, then keyword
bmi = compute_bmi(70, height_m=1.75)
print(f"BMI (mixed): {bmi}")

### 2.3 Default Parameter Values

Set a default so callers can omit that argument. Defaults make functions flexible without making calls verbose.

In [ ]:
def format_currency(amount, currency="₹", decimals=2):
    """Format a number as a currency string."""
    return f"{currency}{amount:,.{decimals}f}"

# Use all defaults
print(format_currency(1299.5))

# Override currency
print(format_currency(49.99, currency="$"))

# Override all
print(format_currency(1500, currency="€", decimals=0))

# In data science: a function that works for any column, with sensible defaults
def fill_missing(values, fill_value=0):
    """Replace None entries with fill_value."""
    return [fill_value if v is None else v for v in values]

data = [88, None, 72, None, 95]
print(fill_missing(data))           # default fill: 0
print(fill_missing(data, fill_value=-1))   # custom fill: -1

### 2.4 Variable Number of Arguments — `*args` and `**kwargs`

When you don't know how many arguments will be passed, use `*args` (tuple of positional args) or `**kwargs` (dict of keyword args).

In [ ]:
# *args: accept any number of positional arguments
def total_score(*subject_scores):
    """Sum scores across any number of subjects."""
    return sum(subject_scores)

print(total_score(85, 72))              # 2 subjects
print(total_score(85, 72, 91))          # 3 subjects
print(total_score(85, 72, 91, 68, 79))  # 5 subjects

# **kwargs: accept any number of keyword arguments as a dict
def create_record(**fields):
    """Build a data record from any keyword arguments."""
    return fields

r = create_record(name="Priya", age=20, city="Mumbai", score=88.5)
print(r)

---
## 3. Return Values

### 3.1 Returning a Single Value

In [ ]:
def square(n):
    return n ** 2

# The return value can be used anywhere a value can be used
result = square(7)              # store it
print(square(5) + square(3))    # use it inline
print([square(i) for i in range(1, 6)])  # use it in a comprehension

### 3.2 Returning Multiple Values

Python lets you return multiple values separated by commas — they come back as a tuple.

In [ ]:
def min_max_mean(values):
    """Return (minimum, maximum, mean) for a list of numbers."""
    valid = [v for v in values if v is not None]
    if not valid:
        return None, None, None
    return min(valid), max(valid), sum(valid) / len(valid)

scores = [85, None, 72, 91, 38, 60, None, 77]

# Option 1: capture the tuple
result = min_max_mean(scores)
print(f"Result tuple: {result}")

# Option 2: unpack directly (most common)
lo, hi, avg = min_max_mean(scores)
print(f"Min: {lo}  Max: {hi}  Mean: {avg:.1f}")

# Option 3: ignore some values with _
_, hi, _  = min_max_mean(scores)
print(f"Max only: {hi}")

In [ ]:
# Applied: a data quality check that returns both the result and a reason
def validate_age(value):
    """Return (is_valid, cleaned_value, message)."""
    if value is None or str(value).strip() == "":
        return False, None, "Missing"
    try:
        age = int(float(value))   # handles "20.0" from CSV
    except (ValueError, TypeError):
        return False, None, f"Not a number: {value!r}"
    if not (0 <= age <= 120):
        return False, None, f"Out of range: {age}"
    return True, age, "OK"

test_inputs = ["20", "25.0", None, "", "abc", "150", "-5", 30]
print(f"{'Input':<10}  {'Valid':<6}  {'Value':<8}  {'Message'}")
print("─" * 42)
for inp in test_inputs:
    ok, val, msg = validate_age(inp)
    print(f"{str(inp):<10}  {str(ok):<6}  {str(val):<8}  {msg}")

### 3.3 Functions That Return Nothing

If a function has no `return` statement (or a bare `return`), it returns `None`. These are typically **side-effect functions** — they print output, write files, or modify external state.

In [ ]:
def print_report(title, data):
    """Print a formatted data report. Returns nothing."""
    print(f"\n{'═' * 40}")
    print(f"  {title}")
    print(f"{'─' * 40}")
    for key, value in data.items():
        print(f"  {key:<16}: {value}")
    print(f"{'═' * 40}")

stats = {"Count": 150, "Mean": 72.4, "Std Dev": 15.2, "Min": 28, "Max": 99}
print_report("Exam Score Summary", stats)

# Show that it returns None
result = print_report("Test", {"a": 1})
print(f"\nreturn value: {result!r}")

---
## 4. Variable Scope — Local vs Global

**Scope** determines which code can see which variables.

- **Local scope:** Variables created inside a function exist only within that function.
- **Global scope:** Variables created outside all functions are accessible everywhere.

Functions create an isolated workspace. This is a feature, not a bug — it means you can reuse variable names like `total`, `i`, `result` inside functions without conflicting with each other.

In [ ]:
# Local variables exist only inside the function
def compute_total(scores):
    total = sum(scores)     # 'total' is LOCAL to this function
    return total

result = compute_total([85, 72, 91])
print(f"Result: {result}")

# 'total' does not exist outside the function
try:
    print(total)    # NameError — total only lived inside compute_total
except NameError as e:
    print(f"NameError: {e}")

In [ ]:
# Functions CAN read global variables (but avoid relying on this)
PASSING_SCORE = 40     # global constant

def has_passed(score):
    return score >= PASSING_SCORE   # reads PASSING_SCORE from global scope

print(has_passed(55))   # True
print(has_passed(35))   # False

# Reading constants from global scope is fine.
# But MODIFYING a global variable from inside a function is dangerous —
# it creates hidden, hard-to-track side effects.

In [ ]:
# ❌ Bad: function secretly modifies a global variable
count = 0

def process_bad(score):
    global count
    count += 1          # modifies global — hidden side effect!
    return score * 2

process_bad(50)
process_bad(75)
print(f"count after 2 calls: {count}")   # 2 — but who changed it?

# ✅ Good: function is pure — returns a value, no side effects
def process_good(score, current_count):
    new_count = current_count + 1   # local variable, no mutation
    return score * 2, new_count     # return both values

count = 0
_, count = process_good(50, count)
_, count = process_good(75, count)
print(f"count after 2 calls: {count}")   # 2 — and it's explicit

---
## 5. Docstrings

A docstring is a string literal immediately after `def`. It documents what the function does, its parameters, and what it returns. Python's built-in `help()` displays it.

In [ ]:
# One-line docstring — use for simple, obvious functions
def square(n):
    """Return the square of n."""
    return n ** 2

# Multi-line docstring — use when parameters or return values need explanation
def compute_stats(values, label="column"):
    """
    Return descriptive statistics for a list of numbers.

    Parameters
    ----------
    values : list
        List of numeric values (None entries are ignored).
    label  : str
        Name of the column / series (used in output only).

    Returns
    -------
    dict with keys: count, missing, mean, median, std, min, max
    """
    valid = [v for v in values if v is not None and isinstance(v, (int, float))]
    if not valid:
        return {"label": label, "count": 0, "missing": len(values)}

    n        = len(valid)
    mean     = sum(valid) / n
    sorted_v = sorted(valid)
    mid      = n // 2
    median   = sorted_v[mid] if n % 2 == 1 else (sorted_v[mid-1] + sorted_v[mid]) / 2
    variance = sum((v - mean) ** 2 for v in valid) / n

    return {
        "label"  : label,
        "count"  : n,
        "missing": len(values) - n,
        "mean"   : round(mean, 2),
        "median" : median,
        "std"    : round(variance ** 0.5, 2),
        "min"    : min(valid),
        "max"    : max(valid),
    }

# Access the docstring
help(compute_stats)

In [ ]:
# Test compute_stats
scores  = [85, None, 72, 91, 38, 60, None, 77, 55, 88]
salaries = [45000, 62000, 38000, 75000, 55000, 90000, 48000]

for stat_dict in [compute_stats(scores, "Exam Scores"),
                  compute_stats(salaries, "Salary (₹)")]:
    print(f"\n{stat_dict['label']}")
    for k, v in stat_dict.items():
        if k != "label":
            print(f"  {k:<8}: {v}")

---
## 6. Lambda Functions — One-Line Functions

A `lambda` is an anonymous (nameless) function written in one line.

```python
lambda parameters: expression
```

Lambdas are used when you need a **simple, short function** that you'll only use once — most often as a `key` for sorting.

In [ ]:
# Lambda equivalent of a one-liner def
double = lambda x: x * 2
print(double(5))   # 10

add = lambda x, y: x + y
print(add(3, 7))   # 10

# Same as:
def add_fn(x, y):
    return x + y

In [ ]:
# The main use case: key= argument in sort() and sorted()

students = [
    {"name": "Priya",  "score": 88, "age": 20},
    {"name": "Rohan",  "score": 72, "age": 22},
    {"name": "Meera",  "score": 95, "age": 19},
    {"name": "Karan",  "score": 61, "age": 21},
    {"name": "Ananya", "score": 88, "age": 20},
]

# Sort by score descending
by_score = sorted(students, key=lambda s: s["score"], reverse=True)
print("Sorted by score (high → low):")
for s in by_score:
    print(f"  {s['name']:<10} {s['score']}")

print()
# Sort by score descending, then by name alphabetically (tie-breaker)
by_score_name = sorted(students, key=lambda s: (-s["score"], s["name"]))
print("Sorted by score desc, then name asc (tie-break):")
for s in by_score_name:
    print(f"  {s['name']:<10} {s['score']}")

In [ ]:
# Lambda with filter() and map() — functional programming style
scores = [85, 42, 72, 95, 61, 38, 80, 55]

# filter(): keep only items where function returns True
passing = list(filter(lambda s: s >= 40, scores))
print(f"Passing: {passing}")

# map(): transform every item
scaled  = list(map(lambda s: round(s * 0.9, 1), scores))   # 10% reduction
print(f"Scaled : {scaled}")

# Note: list comprehensions are usually preferred over filter/map in Python
# The comprehension versions:
passing2 = [s for s in scores if s >= 40]
scaled2  = [round(s * 0.9, 1) for s in scores]
print(f"Same?  : {passing == passing2 and scaled == scaled2}")

---
## 7. Building a Data Science Toolkit

Let's put everything together and build a small library of reusable functions — the kind you'd actually put in a real project.

### 7.1 String Cleaner

In [ ]:
def clean_string(value, default=None, case="title"):
    """
    Clean a string: strip whitespace, apply case transform, handle None/empty.

    case options: 'title', 'upper', 'lower', 'none'
    Returns default if value is None or empty after stripping.
    """
    if value is None:
        return default
    cleaned = str(value).strip()
    if not cleaned:
        return default
    if case == "title": return cleaned.title()
    if case == "upper": return cleaned.upper()
    if case == "lower": return cleaned.lower()
    return cleaned


# Test
messy_names = ["  PRIYA SHARMA  ", "rohan verma", "", None, "  meera PILLAI"]
print("String cleaner:")
for raw in messy_names:
    cleaned = clean_string(raw)
    print(f"  {str(raw):<22}  →  {str(cleaned)!r}")

### 7.2 Safe Type Converter

In [ ]:
def safe_to_number(value, target_type=float, default=None):
    """
    Safely convert value to int or float.
    Returns default if conversion fails or value is missing.
    """
    if value is None or str(value).strip() == "":
        return default
    try:
        return target_type(float(value))   # float() first handles "42.0"
    except (ValueError, TypeError):
        return default


# Test
test_cases = [
    ("88",     int,   None),
    ("72.5",   float, None),
    ("72.5",   int,   None),   # int(float("72.5")) = 72
    (None,     float, 0.0),
    ("",       int,   -1),
    ("abc",    float, None),
    ("₹1299",  float, None),   # can't convert this
]

print(f"{'Input':<12} {'Type':<8} {'Default':<10} {'Result'}")
print("─" * 42)
for val, typ, default in test_cases:
    result = safe_to_number(val, typ, default)
    print(f"{str(val):<12} {typ.__name__:<8} {str(default):<10} {result!r}")

### 7.3 Flexible Classifier

In [ ]:
def classify(value, thresholds, below_label="Unknown"):
    """
    Assign a label to a numeric value using a list of (threshold, label) pairs.
    Thresholds are sorted descending; first match wins.
    Returns below_label if value is below all thresholds.

    Example:
        classify(72, [(80, 'A'), (60, 'B'), (40, 'C'), (0, 'F')]) → 'B'
    """
    if value is None:
        return below_label
    for threshold, label in sorted(thresholds, key=lambda x: x[0], reverse=True):
        if value >= threshold:
            return label
    return below_label


# Grade classification
GRADE_THRESHOLDS = [(80, "A"), (60, "B"), (40, "C"), (0, "F")]

# BMI category
BMI_THRESHOLDS = [(30.0, "Obese"), (25.0, "Overweight"),
                  (18.5, "Normal"), (0.0, "Underweight")]

# Credit score (CIBIL style)
CIBIL_THRESHOLDS = [(750, "Excellent"), (700, "Good"),
                    (650, "Fair"),      (300, "Poor")]

scores  = [95, 82, 74, 61, 45, 32, None]
bmis    = [16.0, 22.5, 27.3, 35.1]
cibils  = [810, 730, 665, 510]

print("Grade classification:")
for s in scores:
    print(f"  {str(s):<5} → {classify(s, GRADE_THRESHOLDS, 'N/A')}")

print("\nBMI classification:")
for b in bmis:
    print(f"  {b:<6} → {classify(b, BMI_THRESHOLDS)}")

print("\nCIBIL classification:")
for c in cibils:
    print(f"  {c:<4} → {classify(c, CIBIL_THRESHOLDS)}")

### 7.4 Row Validator

In [ ]:
def validate_row(row, schema):
    """
    Validate a data row against a schema dict.

    Schema format:
        {"field": {"type": type, "required": bool, "min": num, "max": num,
                   "allowed": list}}

    Returns (is_valid, list_of_issues).
    """
    issues = []

    for field, rules in schema.items():
        value = row.get(field)

        # Required check
        if rules.get("required", False) and (value is None or value == ""):
            issues.append(f"{field}: required but missing")
            continue   # no point checking further rules for missing value

        if value is None:
            continue   # optional and missing — OK

        # Type check
        expected_type = rules.get("type")
        if expected_type and not isinstance(value, expected_type):
            issues.append(f"{field}: expected {expected_type.__name__}, got {type(value).__name__}")
            continue

        # Range check
        if "min" in rules and value < rules["min"]:
            issues.append(f"{field}: {value} < minimum {rules['min']}")
        if "max" in rules and value > rules["max"]:
            issues.append(f"{field}: {value} > maximum {rules['max']}")

        # Allowed values check
        if "allowed" in rules and value not in rules["allowed"]:
            issues.append(f"{field}: '{value}' not in {rules['allowed']}")

    return len(issues) == 0, issues


STUDENT_SCHEMA = {
    "name"       : {"type": str,   "required": True},
    "age"        : {"type": int,   "required": True,  "min": 15, "max": 35},
    "score"      : {"type": (int, float), "min": 0, "max": 100},
    "department" : {"allowed": ["CS", "EC", "ME", "CE", "IT"]},
}

rows = [
    {"name": "Priya",  "age": 20,  "score": 88.5, "department": "CS"},
    {"name": "",       "age": 21,  "score": 72.0, "department": "ME"},  # empty name
    {"name": "Meera",  "age": 19,  "score": 115,  "department": "CS"},  # score > 100
    {"name": "Karan",  "age": 99,  "score": 60.0, "department": "CS"},  # age > 35
    {"name": "Ananya", "age": 22,  "score": 91.0, "department": "BA"},  # bad dept
    {"name": "Dev",    "age": None, "score": None, "department": None},  # missing required
]

print(f"{'Row':<4}  {'Name':<10}  {'Status'}")
print("─" * 60)
for i, row in enumerate(rows, 1):
    ok, issues = validate_row(row, STUDENT_SCHEMA)
    name = row.get("name") or "(empty)"
    if ok:
        print(f"{i:<4}  {name:<10}  ✓ OK")
    else:
        print(f"{i:<4}  {name:<10}  ✗ {'; '.join(issues)}")

### 7.5 Putting It Together — A Mini Data Pipeline

In [ ]:
# Simulate raw CSV data (everything arrives as strings)
raw_data = [
    {"name": "  PRIYA SHARMA  ", "age": "20",  "score": "88.5",  "dept": "CS"},
    {"name": "rohan verma",      "age": "22",  "score": "72",    "dept": "ME"},
    {"name": "",                 "age": "19",  "score": "95.0",  "dept": "CS"},   # invalid
    {"name": "karan mehta",      "age": "abc", "score": "61",    "dept": "IT"},   # bad age
    {"name": "ananya singh",     "age": "22",  "score": "115",   "dept": "EC"},   # bad score
    {"name": "dev patel",        "age": "25",  "score": "77",    "dept": "CS"},
]

# Step 1: Clean and convert (uses our toolkit functions)
def clean_record(raw):
    """Convert a raw CSV record to typed, cleaned values."""
    return {
        "name" : clean_string(raw.get("name")),
        "age"  : safe_to_number(raw.get("age"),   int,   None),
        "score": safe_to_number(raw.get("score"), float, None),
        "dept" : clean_string(raw.get("dept"), case="upper"),
    }

# Step 2: Validate (uses validate_row)
SCHEMA = {
    "name" : {"type": str,  "required": True},
    "age"  : {"type": int,  "required": True, "min": 15, "max": 35},
    "score": {"type": (int, float), "min": 0, "max": 100},
    "dept" : {"allowed": ["CS", "EC", "ME", "CE", "IT"]},
}

# Step 3: Enrich (uses classify)
def enrich_record(record):
    """Add derived columns to a cleaned record."""
    record["grade"] = classify(record["score"], GRADE_THRESHOLDS, "N/A")
    return record

# Run the pipeline
clean_records   = []
rejected_records = []

for raw in raw_data:
    cleaned = clean_record(raw)
    ok, issues = validate_row(cleaned, SCHEMA)
    if ok:
        enriched = enrich_record(cleaned)
        clean_records.append(enriched)
    else:
        rejected_records.append({"raw": raw, "issues": issues})

print(f"Pipeline results: {len(clean_records)} clean, {len(rejected_records)} rejected\n")

print("Clean records:")
print(f"  {'Name':<15} {'Age':>4} {'Score':>6} {'Dept':<6} {'Grade'}")
print("  " + "─" * 42)
for r in clean_records:
    print(f"  {r['name']:<15} {r['age']:>4} {r['score']:>6.1f} {r['dept']:<6} {r['grade']}")

print("\nRejected records:")
for r in rejected_records:
    name = r["raw"].get("name") or "(empty)"
    print(f"  {name:<20} → {'; '.join(r['issues'])}")

---
## ⚠️ Common Mistakes

In [ ]:
# ── Mistake 1: Forgetting return — function silently returns None ─────────────

def get_grade_broken(score):
    if score >= 80: grade = "A"    # stores grade locally
    elif score >= 60: grade = "B"
    else: grade = "C"
    # ❌ Forgot return! grade lives inside the function, never comes out

result = get_grade_broken(88)
print(f"Broken function returned: {result!r}")  # None!

def get_grade_fixed(score):
    if score >= 80: return "A"     # ✅ return sends value back
    elif score >= 60: return "B"
    else: return "C"

print(f"Fixed function returned:  {get_grade_fixed(88)!r}")

In [ ]:
# ── Mistake 2: Printing instead of returning — can't use the result ───────────

def double_print(n):
    print(n * 2)    # ❌ prints to screen, returns None

def double_return(n):
    return n * 2    # ✅ returns value, caller decides what to do with it

# Problem: can't use the print version in a calculation
result = double_print(5)    # prints 10
print(f"print version: can I add 1? {result} + 1 = ", end="")
try:
    print(result + 1)       # TypeError — result is None
except TypeError as e:
    print(f"TypeError: {e}")

# Return version: fully flexible
val = double_return(5)
print(f"return version: {val} + 1 = {val + 1}")
print([double_return(i) for i in range(5)])

In [ ]:
# ── Mistake 3: Mutable default argument — Python's sneakiest gotcha ───────────

# ❌ The list [] is created ONCE when the function is defined, not at each call
def add_score_bad(score, results=[]):
    results.append(score)     # mutates the shared default list!
    return results

print(add_score_bad(88))     # [88]
print(add_score_bad(72))     # Expected: [72] — Got: [88, 72]  ← bug!
print(add_score_bad(95))     # Expected: [95] — Got: [88, 72, 95] ← bug!

print()
# ✅ Fix: use None as default, create new list inside the function
def add_score_good(score, results=None):
    if results is None:
        results = []          # new list on every call where results=None
    results.append(score)
    return results

print(add_score_good(88))    # [88]
print(add_score_good(72))    # [72]  ← correct
print(add_score_good(95))    # [95]  ← correct

print()
print("Rule: NEVER use a mutable object (list, dict, set) as a default argument.")
print("      Use None and create the object inside the function.")

In [ ]:
# ── Mistake 4: Calling before defining ───────────────────────────────────────

# ❌ This fails — Python reads top to bottom; the function doesn't exist yet
try:
    result = greet_undefined("Priya")   # NameError
except NameError as e:
    print(f"NameError: {e}")

# ✅ Define first, call after
def greet(name):
    return f"Hello, {name}!"

print(greet("Priya"))

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Calculator Functions

Write four functions: `add`, `subtract`, `multiply`, `divide`. The `divide` function should return `None` (not crash) when the divisor is 0.

In [ ]:
# Your four functions here

# Test cases — all should run without errors
tests = [
    ("add",      10,  3,   13),
    ("subtract", 10,  3,    7),
    ("multiply", 10,  3,   30),
    ("divide",   10,  3,   10/3),
    ("divide",   10,  0,   None),   # divide by zero → None
]

fn_map = {"add": add, "subtract": subtract,
          "multiply": multiply, "divide": divide}

print(f"{'Op':<10} {'a':>4} {'b':>4} {'Expected':>10} {'Got':>10} {'Pass?'}")
print("─" * 48)
for op, a, b, expected in tests:
    result = fn_map[op](a, b)
    passed = result == expected or (result is None and expected is None)
    print(f"{op:<10} {a:>4} {b:>4} {str(expected):>10} {str(result):>10} {'✓' if passed else '✗'}")

### Exercise 2 — Temperature Converter

Write `convert_temperature(value, from_unit, to_unit)` that converts between Celsius (`"C"`), Fahrenheit (`"F"`), and Kelvin (`"K"`).

Formulas:
- C → F: `(C × 9/5) + 32`
- F → C: `(F − 32) × 5/9`
- C → K: `C + 273.15`
- K → C: `K − 273.15`
- F → K: convert via C

In [ ]:
def convert_temperature(value, from_unit, to_unit):
    """Convert temperature between C, F, and K."""
    # Your code here
    pass

# Tests
tests = [
    (0,      "C", "F",  32.0),
    (100,    "C", "F",  212.0),
    (32,     "F", "C",  0.0),
    (0,      "C", "K",  273.15),
    (300,    "K", "C",  26.85),
    (98.6,   "F", "K",  310.15),
    (37,     "C", "C",  37.0),    # same unit
]

print(f"{'Value':>8} {'From':<5} {'To':<5} {'Expected':>10} {'Got':>10} {'Pass?'}")
print("─" * 50)
for val, frm, to, expected in tests:
    result = convert_temperature(val, frm, to)
    if result is not None:
        passed = abs(result - expected) < 0.01
        print(f"{val:>8} {frm:<5} {to:<5} {expected:>10.2f} {result:>10.2f} {'✓' if passed else '✗'}")
    else:
        print(f"{val:>8} {frm:<5} {to:<5} {expected:>10.2f} {'None':>10} ✗")

### Exercise 3 — Column Analyser

Write `analyse_column(data, column_name)` that:
1. Extracts the named column from a list of dicts
2. Separates numeric vs string vs None values
3. Returns a summary dict with counts, and statistics for numeric values

Use the `compute_stats` function defined earlier.

In [ ]:
def analyse_column(data, column_name):
    """
    Analyse a column from a list of dict records.
    Returns a summary dict.
    """
    # Your code here
    pass


# Test dataset
dataset = [
    {"name": "Priya",  "score": 88.5, "city": "Mumbai"},
    {"name": "Rohan",  "score": None, "city": "Delhi"},
    {"name": "Meera",  "score": 95.0, "city": "Mumbai"},
    {"name": None,     "score": 72.0, "city": "Pune"},
    {"name": "Karan",  "score": 61.0, "city": None},
    {"name": "Ananya", "score": 38.0, "city": "Delhi"},
    {"name": "Dev",    "score": 77.0, "city": "Bangalore"},
]

# Analyse each column
for col in ["name", "score", "city"]:
    result = analyse_column(dataset, col)
    if result:
        print(f"\n{col}:")
        for k, v in result.items():
            print(f"  {k:<12}: {v}")

### Exercise 4 — Chained Pipeline

Build a `run_pipeline(raw_records, schema, grade_thresholds)` function that:
1. Cleans each record using `clean_string` and `safe_to_number`
2. Validates each record using `validate_row`
3. Enriches valid records with a `grade` column using `classify`
4. Returns `(clean_records, rejected_records, summary_stats)`

The `summary_stats` should include: total rows, clean count, rejected count, and stats on the `score` column of clean records.

In [ ]:
def run_pipeline(raw_records, schema, grade_thresholds):
    """
    Full data pipeline: clean → validate → enrich → summarise.
    Returns (clean_records, rejected, summary).
    """
    # Your code here — chain together functions from the toolkit
    pass


# Test data
raw = [
    {"name": "  PRIYA  ", "age": "20", "score": "88",  "dept": "CS"},
    {"name": "rohan",     "age": "22", "score": "72",  "dept": "ME"},
    {"name": "",          "age": "19", "score": "95",  "dept": "CS"},   # invalid
    {"name": "meera",     "age": "21", "score": "abc", "dept": "EC"},   # bad score
    {"name": "karan",     "age": "25", "score": "61",  "dept": "IT"},
]

schema = {
    "name" : {"type": str,  "required": True},
    "age"  : {"type": int,  "required": True, "min": 15, "max": 35},
    "score": {"type": (int, float), "min": 0, "max": 100},
    "dept" : {"allowed": ["CS", "EC", "ME", "CE", "IT"]},
}

result = run_pipeline(raw, schema, GRADE_THRESHOLDS)
if result:   # only display if your function returns something
    clean, rejected, summary = result
    print("Summary:", summary)
    print("\nClean records:")
    for r in clean:
        print(f"  {r}")
    print("\nRejected:")
    for r in rejected:
        print(f"  {r}")

### Exercise 5 — (Challenge) Memoisation

**Memoisation** is a technique where a function caches its results so it doesn't recompute the same input twice. It's used heavily in ML for expensive feature computations.

Write a `memoised_stats(values_tuple)` function that:
- Takes a **tuple** of numbers (tuples are hashable, lists are not)
- Computes mean and std dev
- Stores the result in a cache dict
- Returns the cached result on repeated calls without recomputing
- Tracks how many times the cache is hit vs computed

In [ ]:
# Global cache — persists across calls
_stats_cache   = {}
_cache_hits    = 0
_cache_misses  = 0

def memoised_stats(values_tuple):
    """
    Return (mean, std_dev) for values_tuple.
    Uses a cache to avoid recomputing known inputs.
    """
    global _cache_hits, _cache_misses
    # Your code here
    pass

def cache_report():
    total = _cache_hits + _cache_misses
    if total == 0: return
    print(f"Cache: {_cache_hits} hits / {_cache_misses} misses "
          f"({_cache_hits/total*100:.0f}% hit rate) — {len(_stats_cache)} unique inputs stored")

# Test
a = (85, 72, 91, 60, 77)
b = (45, 55, 60, 70, 80)

print(memoised_stats(a))   # computed
print(memoised_stats(b))   # computed
print(memoised_stats(a))   # cache hit — same as first call
print(memoised_stats(a))   # cache hit
print(memoised_stats(b))   # cache hit

cache_report()
# Expected: 2 misses, 3 hits, 2 unique inputs stored

---
## 📌 Key Takeaways

- **A function is a promise: given the same input, always produce the same output.** Functions that rely on hidden global state or have side effects break this promise and become hard to test and debug. Keep functions pure wherever possible.

- **Never use a mutable object (list, dict, set) as a default parameter.** Use `None` and create the object inside. This is the most common intermediate-level Python bug, and it causes silent wrong behaviour — no error message to help you find it.

- **`return` vs `print` is not a style choice — it changes what you can do with the result.** A function that prints its output can't be used in a calculation, comprehension, or pipeline. Return the value; let the caller decide whether to print it.

---

## What's Next?

**Lesson 05 — Data Structures: Lists, Tuples, Sets, Dictionaries**  
You've used lists and dicts throughout these lessons. Next you'll understand all four built-in collections in depth — when to use each, their performance trade-offs, and the operations that make them powerful for storing and manipulating datasets.